# Step 8 ‐ EoMT Mask-Based Anomaly Baselines
**Methods:** MSP · Max Logit · Max Entropy · RbA · Temperature Scaling

**Checkpoints:** EoMT-COCO · EoMT-Cityscapes · EoMT-Finetuned

**Runtime → T4 GPU before running!**

## 0 ‐ GPU Check

Check that a CUDA GPU is available; raise an error if none is detected.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU required — go to Runtime → Change runtime type → T4")

## 1 ‐ Environment Setup

Clone the repository, install lightweight dependencies (timm, einops, scikit-learn), and define paths to checkpoints and the anomaly dataset.

In [ ]:
import subprocess, os, shutil

# ── 1. Clone repo ────────────────────────────────────────────────────
LOCAL_REPO = '/kaggle/working/repo'
EOMT_CODE  = '/kaggle/working/eomt_code'

if not os.path.isdir(LOCAL_REPO):
    subprocess.run(["git", "clone",
        "https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject.git",
        LOCAL_REPO], check=True)

if not os.path.isdir(EOMT_CODE):
    shutil.copytree(f'{LOCAL_REPO}/eomt', EOMT_CODE)
    print(f"✓ EoMT code copied to {EOMT_CODE}")

# ── 2. Installa SOLO pacchetti leggeri (NON toccano torch) ───────────
subprocess.run(["pip", "install", "-q", "timm", "einops", "scikit-learn"], check=False)

# ── 3. Paths ─────────────────────────────────────────────────────────
RESULTS_DIR     = '/kaggle/working/results/eomt'
os.makedirs(RESULTS_DIR, exist_ok=True)

CKPT_COCO       = '/kaggle/input/datasets/federicoremy/eomt-city/eomt_coco.bin'
CKPT_CITYSCAPES = '/kaggle/input/datasets/federicoremy/eomt-city/eomt_cityscapes.bin'
CKPT_FINETUNED  = '/kaggle/input/datasets/federicoremy/eomt-finetuned/eomt_ft_head_epoch04.ckpt'
ANOMALY_DATA    = '/kaggle/input/datasets/federicoremy/anomaly-dataset/Validation_Dataset'

print("\n=== Checkpoint check ===")
for name, path in [("COCO", CKPT_COCO), ("Cityscapes", CKPT_CITYSCAPES), ("Finetuned", CKPT_FINETUNED)]:
    print(f"  {name:12s}: {'✓' if os.path.exists(path) else '✗ NOT FOUND'}  {path}")

print(f"\n=== Anomaly datasets ===")
if os.path.isdir(ANOMALY_DATA):
    for d in sorted(os.listdir(ANOMALY_DATA)):
        if os.path.isdir(os.path.join(ANOMALY_DATA, d)):
            print(f"  {d}/")
else:
    print(f"  ✗ NOT FOUND: {ANOMALY_DATA}")

## 2 ‐ Import EoMT Modules

Import EoMT modules and verify that the `EoMT` and `ViT` classes can be imported successfully.

In [ ]:
import sys
sys.path.insert(0, EOMT_CODE)

import torch
from torchvision import transforms
from PIL import Image

device = torch.device('cuda')

# ── Try loading via the Lightning module wrapper ─────────────────────
try:
    from training.mask_classification_semantic import MaskClassificationSemantic
    HAS_LIGHTNING_WRAPPER = True
    print("✓ MaskClassificationSemantic available")
except ImportError:
    HAS_LIGHTNING_WRAPPER = False
    print("✗ Lightning wrapper not found — will use raw model")

# ── Also check raw model ─────────────────────────────────────────────
try:
    from models.eomt import EoMT
    from models.vit import ViT
    print("✓ EoMT + ViT importable")
except Exception as e:
    print(f"Import error: {e}")

# ── Check configs for default model settings ─────────────────────────
print("\n=== Available configs ===")
!ls configs/ 2>/dev/null || echo "No configs dir"

print("\n=== Training module files ===")
!ls training/ 2>/dev/null || echo "No training dir"

print("\n=== Model files ===")
!ls models/

## 3 ‐ Model Loading Utilities

Define `load_eomt` (loads a `.ckpt` or `.bin` checkpoint, auto detecting architecture parameters) and `make_preprocess` (image preprocessing pipeline).

In [ ]:
import sys, os, glob, yaml, types
sys.path.insert(0, EOMT_CODE)

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms

device = torch.device('cuda')

# ── Model loading ────────────────────────────────────────────────────
def load_eomt(ckpt_path):
    """
    Load an EoMT model from a checkpoint.
    Extracts state_dict from both .ckpt and .bin, then builds the model.
    """

    # --- Lightning .ckpt: extract state_dict manually ---
    if ckpt_path.endswith('.ckpt'):
        # Monkey-patch missing timm modules so pickle can unpickle the checkpoint
        for fake_module in ['timm.layers.attention', 'timm.layers.attention2']:
            if fake_module not in sys.modules:
                parts = fake_module.split('.')
                for i in range(len(parts)):
                    mod_path = '.'.join(parts[:i+1])
                    if mod_path not in sys.modules:
                        sys.modules[mod_path] = types.ModuleType(mod_path)

        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            state = ckpt['state_dict']
            print(f"  Loaded .ckpt as raw state_dict ({len(state)} keys)")
        else:
            state = ckpt

    # --- Raw .bin / .pth ---
    else:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        if isinstance(ckpt, dict):
            state = ckpt.get('state_dict', ckpt.get('model', ckpt))
        else:
            state = ckpt

    # Strip 'network.' prefix (saved from Lightning wrapper)
    cleaned = {}
    for k, v in state.items():
        new_k = k
        for prefix in ['network.', 'model.', 'module.']:
            if new_k.startswith(prefix):
                new_k = new_k[len(prefix):]
        cleaned[new_k] = v

    # Detect architecture params from state_dict shapes
    num_classes_plus1 = cleaned['class_head.weight'].shape[0]
    num_classes = num_classes_plus1 - 1
    print(f"  num_classes = {num_classes} (class_head output = {num_classes_plus1})")

    num_q = cleaned['q.weight'].shape[0]
    print(f"  num_q = {num_q}")

    num_blocks = cleaned['attn_mask_probs'].shape[0]
    print(f"  num_blocks = {num_blocks}")

    # Detect img_size from pos_embed shape
    num_patches = cleaned['encoder.backbone.pos_embed'].shape[1]
    grid_size = int(num_patches ** 0.5)
    img_size = grid_size * 16
    print(f"  img_size = {img_size}x{img_size} (from pos_embed: {num_patches} patches)")

    # Build model
    encoder = ViT(
        img_size=(img_size, img_size),
        backbone_name='vit_base_patch14_reg4_dinov2',
    )
    model = EoMT(
        encoder=encoder,
        num_classes=num_classes,
        num_q=num_q,
        num_blocks=num_blocks,
    )
    model.load_state_dict(cleaned, strict=False)
    print(f"  ✓ Loaded: {os.path.basename(ckpt_path)}")
    return model.to(device).eval()


# ── Preprocessing (dynamic per-model) ────────────────────────────────
def make_preprocess(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("✓ Model loading utilities defined")

## 4 ‐ Verify Model Output Format

Load the COCO checkpoint onto the GPU, run a forward pass on a dummy input, and print output shapes to confirm the model output format.

In [ ]:
# Load the COCO checkpoint to understand the output format
test_model = load_eomt(CKPT_COCO)

# Create a dummy input
dummy = torch.randn(1, 3, 640, 640, device=device)
with torch.no_grad():
    out = test_model(dummy)

# Inspect output
print(f"\nOutput type: {type(out)}")
if isinstance(out, dict):
    for k, v in out.items():
        if isinstance(v, torch.Tensor):
            print(f"  '{k}': shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  '{k}': type={type(v)}")
elif isinstance(out, (tuple, list)):
    for i, v in enumerate(out):
        if isinstance(v, torch.Tensor):
            print(f"  [{i}]: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  [{i}]: type={type(v)}")
else:
    print(f"  shape={out.shape}")

del test_model
torch.cuda.empty_cache()
print("\n✓ Output format understood — proceed to Cell 5")

## 5 ‐ Anomaly Scoring Functions

Define the four anomaly scoring functions  MSP, Max Logit, Max Entropy, and RbA all based on combining percquery class logits and mask logits.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# ── Extract model outputs ────────────────────────────────────────────
def get_model_outputs(model, inp):
    """
    Run model and return (class_logits, mask_logits).
    EoMT returns (mask_logits_per_layer, class_logits_per_layer).
    We take the last layer's output for inference.
    """
    with torch.no_grad():
        out = model(inp)

    # EoMT output: (list_of_mask_logits, list_of_class_logits)
    mask_logits_list, class_logits_list = out
    mask_logits = mask_logits_list[-1]   # last layer: [B, Q, H', W']
    class_logits = class_logits_list[-1] # last layer: [B, Q, C+1]

    return class_logits, mask_logits


# ── Combine class + mask into per-pixel logits ───────────────────────
def get_pixel_probs(class_logits, mask_logits, gt_size, temperature=1.0):
    """
    Combine query-level class logits and mask logits into per-pixel
    class probabilities.

    Returns:
        pixel_probs: [H, W, C] numpy array (C = num known classes, no void)
        mask_probs:  [B, Q, H, W] tensor (for RbA)
    """
    # Apply temperature to class logits
    cl = class_logits / temperature  # [B, Q, C+1]

    # Class probabilities (exclude the last "no-object" class for known classes)
    class_probs = F.softmax(cl, dim=-1)         # [B, Q, C+1]

    # Mask probabilities
    mp = torch.sigmoid(mask_logits)              # [B, Q, H', W']
    mp = F.interpolate(mp, size=gt_size, mode='bilinear', align_corners=False)

    # Per-pixel class distribution: einsum over queries
    # pixel_probs[b,c,h,w] = sum_q class_probs[b,q,c] * mask_probs[b,q,h,w]
    pixel_probs = torch.einsum('bqc,bqhw->bchw', class_probs, mp)

    # Normalize so probabilities sum to 1 per pixel
    pixel_probs = pixel_probs / (pixel_probs.sum(dim=1, keepdim=True) + 1e-8)

    return pixel_probs, mp


# ══════════════════════════════════════════════════════════════════════
# ANOMALY SCORING FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

def score_msp(class_logits, mask_logits, gt_size, temperature=1.0):
    """
    MSP (Maximum Softmax Probability).
    anomaly_score = 1 - max_c pixel_probs[c]
    """
    pixel_probs, _ = get_pixel_probs(class_logits, mask_logits, gt_size, temperature)
    msp = pixel_probs.max(dim=1).values  # [B, H, W]
    return (1.0 - msp)[0].cpu().numpy()


def score_maxlogit(class_logits, mask_logits, gt_size, temperature=1.0):
    """
    Max Logit.
    Use the raw logit (not probability) of the most confident class.
    anomaly_score = -max_c pixel_logits[c]
    """
    cl = class_logits / temperature
    mp = torch.sigmoid(mask_logits)
    mp = F.interpolate(mp, size=gt_size, mode='bilinear', align_corners=False)

    # Per-pixel logits (unnormalized)
    pixel_logits = torch.einsum('bqc,bqhw->bchw', cl, mp)
    max_logit = pixel_logits.max(dim=1).values  # [B, H, W]
    return (-max_logit)[0].cpu().numpy()


def score_maxentropy(class_logits, mask_logits, gt_size, temperature=1.0):
    """
    Max Entropy.
    anomaly_score = H(pixel_probs) = -Σ_c p_c log(p_c)
    """
    pixel_probs, _ = get_pixel_probs(class_logits, mask_logits, gt_size, temperature)
    entropy = -(pixel_probs * (pixel_probs + 1e-8).log()).sum(dim=1)  # [B, H, W]
    return entropy[0].cpu().numpy()


def score_rba(class_logits, mask_logits, gt_size, temperature=1.0):
    """
    RbA (Rejected by All) — Nayal et al., 2022.

    A pixel is anomalous if NO query claims it as a known class.
    For each query q:
      claim_q = σ(mask_logits_q) * (1 - P(no-object | q))
    anomaly_score = 1 - max_q claim_q

    The last class dimension in class_logits is the "no-object" class.
    """
    cl = class_logits / temperature  # [B, Q, C+1]
    class_probs = F.softmax(cl, dim=-1)           # [B, Q, C+1]

    # "Known class" confidence = 1 - P(no-object)
    # The last class is "no-object" / void
    known_conf = 1.0 - class_probs[:, :, -1]      # [B, Q]
    known_conf = known_conf.unsqueeze(-1).unsqueeze(-1)  # [B, Q, 1, 1]

    # Mask probabilities
    mp = torch.sigmoid(mask_logits)                # [B, Q, H', W']
    mp = F.interpolate(mp, size=gt_size, mode='bilinear', align_corners=False)

    # Each query's "claim" on each pixel
    query_claims = mp * known_conf                 # [B, Q, H, W]

    # Pixel is anomalous if no query claims it
    max_claim = query_claims.max(dim=1).values     # [B, H, W]
    return (1.0 - max_claim)[0].cpu().numpy()


METHODS = {
    'MSP':        score_msp,
    'MaxLogit':   score_maxlogit,
    'MaxEntropy': score_maxentropy,
    'RbA':        score_rba,
}

print("✓ Anomaly scoring functions defined: MSP, MaxLogit, MaxEntropy, RbA")

## 6 ‐ Evaluation Metrics & Dataset Loader

Implement the AuPRC and FPR95 metrics, then auto discover (image, ground-truth) pairs across the five benchmark datasets.

In [ ]:
from sklearn.metrics import average_precision_score, roc_curve
from pathlib import Path


def compute_auprc(scores, gt):
    """
    Area Under Precision-Recall Curve.
    GT convention: 0 = inlier, 255 = void (ignored), anything else = anomaly.
    """
    s = scores.flatten()
    g = gt.flatten().astype(np.int64)
    valid = (g != 255)
    if valid.sum() == 0:
        return float('nan')
    # Binarize: 0 = inlier, >0 and !=255 = anomaly
    g_bin = (g[valid] > 0).astype(np.int64)
    if g_bin.max() == 0 or g_bin.min() == g_bin.max():
        return float('nan')
    return average_precision_score(g_bin, s[valid])


def compute_fpr95(scores, gt):
    """
    False Positive Rate at 95% True Positive Rate.
    """
    s = scores.flatten()
    g = gt.flatten().astype(np.int64)
    valid = (g != 255)
    if valid.sum() == 0:
        return float('nan')
    g_bin = (g[valid] > 0).astype(np.int64)
    if g_bin.max() == 0:
        return float('nan')
    fpr, tpr, _ = roc_curve(g_bin, s[valid])
    idx = min(np.searchsorted(tpr, 0.95), len(fpr) - 1)
    return float(fpr[idx])


# ── Dataset auto-discovery ───────────────────────────────────────────
# The anomaly dataset folder names may vary. We try several known patterns.

DATASET_ALIASES = {
    'SMIYC_RA21':  ['RoadAnomaly21'],
    'SMIYC_RO21':  ['RoadObsticle21', 'RoadObstacle21'],
    'FS_LF':       ['FS_LostFound_full', 'LostAndFound', 'FS_Lost_Found'],
    'FS_Static':   ['fs_static', 'FishyscapesStatic', 'FS_Static'],
    'RoadAnomaly': ['RoadAnomaly'],
}


def find_dataset_path(base_dir, ds_name):
    """Find the actual folder for a dataset, trying known aliases."""
    for alias in DATASET_ALIASES.get(ds_name, [ds_name]):
        candidate = os.path.join(base_dir, alias)
        if os.path.isdir(candidate):
            return candidate
    return None


def get_image_gt_pairs(dataset_path):
    """
    Discover (image, ground_truth) pairs in a dataset folder.
    Handles multiple common directory structures:
      - images_path/ + labels_path/ (with matching stems)
      - image + image_labels_semantic.png side by side
      - image + image_gt.png side by side
    """
    dpath = Path(dataset_path)
    pairs = []

    # Strategy 1: look for dedicated images/ and labels/ folders
    for img_dir_name in ['images', 'original', 'leftImg8bit']:
        for gt_dir_name in ['labels_masks', 'labels', 'gt', 'masks', 'ground_truth']:
            img_dir = dpath / img_dir_name
            gt_dir  = dpath / gt_dir_name
            if img_dir.is_dir() and gt_dir.is_dir():
                for img_path in sorted(img_dir.glob('*')):
                    if img_path.suffix.lower() not in ('.jpg', '.png', '.jpeg', '.webp'):
                        continue
                    stem = img_path.stem
                    for gt_pattern in [
                        gt_dir / f'{stem}.png',
                        gt_dir / f'{stem}_labels_semantic.png',
                        gt_dir / f'{stem}_gt.png',
                        gt_dir / f'{stem}_mask.png',
                    ]:
                        if gt_pattern.exists():
                            pairs.append((str(img_path), str(gt_pattern)))
                            break
                if pairs:
                    return pairs

    # Strategy 2: image and label side by side
    all_imgs = sorted(dpath.rglob('*.jpg')) + sorted(dpath.rglob('*.png')) + sorted(dpath.rglob('*.jpeg'))
    for img_path in all_imgs:
        name = str(img_path).lower()
        if any(x in name for x in ['label', '_gt', 'mask', 'ground']):
            continue
        stem = img_path.stem
        parent = img_path.parent
        for gt in [
            parent / f'{stem}_labels_semantic.png',
            parent / f'{stem}_gt.png',
            parent / f'{stem}_mask.png',
            parent.parent / 'labels' / f'{stem}.png',
            parent.parent / 'labels_masks' / f'{stem}.png',
            parent.parent / 'gt' / f'{stem}.png',
            parent.parent / 'masks' / f'{stem}.png',
        ]:
            if gt.exists():
                pairs.append((str(img_path), str(gt)))
                break

    return pairs


# ── Discover all datasets ────────────────────────────────────────────
DATASETS = {}
print("=== Dataset discovery ===")
for ds_name in ['SMIYC_RA21', 'SMIYC_RO21', 'FS_LF', 'FS_Static', 'RoadAnomaly']:
    path = find_dataset_path(ANOMALY_DATA, ds_name)
    if path:
        pairs = get_image_gt_pairs(path)
        DATASETS[ds_name] = {'path': path, 'pairs': pairs}
        print(f"  ✓ {ds_name:15s}: {len(pairs):3d} pairs  →  {path}")
    else:
        print(f"  ✗ {ds_name:15s}: NOT FOUND in {ANOMALY_DATA}")
        # List available folders to help debug
        if os.path.isdir(ANOMALY_DATA):
            avail = os.listdir(ANOMALY_DATA)
            print(f"    Available: {avail}")

print(f"\n✓ Found {len(DATASETS)} datasets")

## 7 ‐ Main Evaluation Loop

Main evaluation loop: for each checkpoint × dataset × method, compute anomaly scores and metrics; also runs temperature scaling on MSP by reusing cached logits to avoid redundant forward passes.

In [ ]:
import json, time

# ── Configuration ────────────────────────────────────────────────────
CHECKPOINTS = {}
for name, path in [
    ('COCO', CKPT_COCO),
    ('Cityscapes', CKPT_CITYSCAPES),
    ('Finetuned', CKPT_FINETUNED),
]:
    if os.path.exists(path):
        CHECKPOINTS[name] = path
    else:
        print(f"⚠ Skipping {name} — not found: {path}")

TEMPERATURES = [0.5, 0.75, 1.1]

DS_NAMES = ['SMIYC_RA21', 'SMIYC_RO21', 'FS_LF', 'FS_Static', 'RoadAnomaly']

# ── Results storage ──────────────────────────────────────────────────
all_results = {}

for ckpt_name, ckpt_path in CHECKPOINTS.items():
    print(f"\n{'#'*60}")
    print(f"# Loading: EoMT-{ckpt_name}")
    print(f"{'#'*60}")

    t0 = time.time()
    model = load_eomt(ckpt_path)
    print(f"  Model loaded in {time.time()-t0:.1f}s")

    # Detect img_size for this checkpoint
    ckpt_tmp = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_tmp = ckpt_tmp.get('state_dict', ckpt_tmp.get('model', ckpt_tmp))
    model_img_size = 640
    for k, v in state_tmp.items():
        if 'pos_embed' in k:
            grid = int(v.shape[1] ** 0.5)
            model_img_size = grid * 16
            break
    del ckpt_tmp, state_tmp
    preprocess = make_preprocess(model_img_size)
    print(f"  Using img_size = {model_img_size}x{model_img_size}")

    all_results[ckpt_name] = {}
    saved_logits = {}

    for ds_name in DS_NAMES:
        if ds_name not in DATASETS:
            print(f"\n  ⚠ {ds_name} not available — skipping")
            continue

        pairs = DATASETS[ds_name]['pairs']
        print(f"\n  === {ds_name} ({len(pairs)} images) ===")

        ds_results = {m: {'auprc': [], 'fpr95': []} for m in METHODS}
        saved_logits[ds_name] = []

        for i, (img_path, gt_path) in enumerate(pairs):
            img = Image.open(img_path).convert('RGB')
            inp = preprocess(img).unsqueeze(0).to(device)

            gt = np.array(Image.open(gt_path))
            gt_size = gt.shape[:2]

            with torch.no_grad():
                class_logits, mask_logits = get_model_outputs(model, inp)

            saved_logits[ds_name].append((
                class_logits.cpu(),
                mask_logits.cpu() if mask_logits is not None else None,
                gt
            ))

            for method_name, score_fn in METHODS.items():
                try:
                    scores = score_fn(class_logits, mask_logits, gt_size)
                    if scores is None:
                        continue

                    s_min, s_max = scores.min(), scores.max()
                    if s_max > s_min:
                        scores = (scores - s_min) / (s_max - s_min)

                    auprc = compute_auprc(scores, gt)
                    fpr95 = compute_fpr95(scores, gt)

                    if not np.isnan(auprc):
                        ds_results[method_name]['auprc'].append(auprc)
                    if not np.isnan(fpr95):
                        ds_results[method_name]['fpr95'].append(fpr95)

                except Exception as e:
                    if i == 0:
                        print(f"    ⚠ {method_name} error: {e}")

            if (i + 1) % 10 == 0:
                print(f"    Processed {i+1}/{len(pairs)} images")

        all_results[ckpt_name][ds_name] = {}
        print(f"\n  {'Method':<15} {'AuPRC':>8} {'FPR95':>8}")
        print(f"  {'-'*33}")
        for m in METHODS:
            if ds_results[m]['auprc']:
                a = np.mean(ds_results[m]['auprc']) * 100
                f = np.mean(ds_results[m]['fpr95']) * 100
                all_results[ckpt_name][ds_name][m] = {'AuPRC': round(a, 2), 'FPR95': round(f, 2)}
                print(f"  {m:<15} {a:>7.2f}% {f:>7.2f}%")
            else:
                print(f"  {m:<15} {'N/A':>8} {'N/A':>8}")

    # ── Temperature scaling (MSP only, per-dataset) ──────────────────
    print(f"\n  === Temperature Scaling (MSP) for EoMT-{ckpt_name} ===")
    all_results[ckpt_name]['temperature'] = {}

    for ds_name in DS_NAMES:
        if ds_name not in saved_logits:
            continue
        logits_list = saved_logits[ds_name]
        all_results[ckpt_name]['temperature'][ds_name] = {}

        for T in TEMPERATURES:
            t_auprc, t_fpr95 = [], []
            for (cl, ml, gt) in logits_list:
                cl_dev = cl.to(device)
                ml_dev = ml.to(device) if ml is not None else None
                gt_size = gt.shape[:2]

                scores = score_msp(cl_dev, ml_dev, gt_size, temperature=T)
                s_min, s_max = scores.min(), scores.max()
                if s_max > s_min:
                    scores = (scores - s_min) / (s_max - s_min)

                auprc = compute_auprc(scores, gt)
                fpr95 = compute_fpr95(scores, gt)
                if not np.isnan(auprc):
                    t_auprc.append(auprc)
                if not np.isnan(fpr95):
                    t_fpr95.append(fpr95)

            if t_auprc:
                a = np.mean(t_auprc) * 100
                f = np.mean(t_fpr95) * 100
                all_results[ckpt_name]['temperature'][ds_name][f'T={T}'] = {
                    'AuPRC': round(a, 2), 'FPR95': round(f, 2)
                }

    # Print temperature results
    for ds_name in DS_NAMES:
        if ds_name in all_results[ckpt_name].get('temperature', {}):
            t_res = all_results[ckpt_name]['temperature'][ds_name]
            if t_res:
                print(f"\n  {ds_name}:")
                for t_key, vals in t_res.items():
                    print(f"    {t_key:<8} AuPRC={vals['AuPRC']:.2f}%  FPR95={vals['FPR95']:.2f}%")

    # Find best T per dataset (by AuPRC) + compute FPR95 at best T
    for ds_name in DS_NAMES:
        if ds_name not in saved_logits:
            continue
        logits_list = saved_logits[ds_name]
        best_T, best_auprc = 1.0, -1
        for T_cand in np.arange(0.3, 3.0, 0.05):
            t_auprc = []
            for (cl, ml, gt) in logits_list:
                cl_dev = cl.to(device)
                ml_dev = ml.to(device) if ml is not None else None
                scores = score_msp(cl_dev, ml_dev, gt.shape[:2], temperature=T_cand)
                s_min, s_max = scores.min(), scores.max()
                if s_max > s_min:
                    scores = (scores - s_min) / (s_max - s_min)
                a = compute_auprc(scores, gt)
                if not np.isnan(a):
                    t_auprc.append(a)
            if t_auprc:
                avg = np.mean(t_auprc)
                if avg > best_auprc:
                    best_auprc = avg
                    best_T = T_cand

        # recompute FPR95 at the best_T found
        best_fpr95 = []
        for (cl, ml, gt) in logits_list:
            cl_dev = cl.to(device)
            ml_dev = ml.to(device) if ml is not None else None
            scores = score_msp(cl_dev, ml_dev, gt.shape[:2], temperature=best_T)
            s_min, s_max = scores.min(), scores.max()
            if s_max > s_min:
                scores = (scores - s_min) / (s_max - s_min)
            f = compute_fpr95(scores, gt)
            if not np.isnan(f):
                best_fpr95.append(f)
        fpr_val = round(np.mean(best_fpr95) * 100, 2) if best_fpr95 else None

        all_results[ckpt_name]['temperature'].setdefault(ds_name, {})[f'best_T'] = {
            'T': round(best_T, 2),
            'AuPRC': round(best_auprc * 100, 2),
            'FPR95': fpr_val
        }
        print(f"  {ds_name} best T={best_T:.2f} → AuPRC={best_auprc*100:.2f}% FPR95={fpr_val}%")

    # Save intermediate
    with open(f'{RESULTS_DIR}/eomt_{ckpt_name}_results.json', 'w') as fp:
        json.dump(all_results[ckpt_name], fp, indent=2)
    print(f"\n  ✓ Results saved for EoMT-{ckpt_name}")

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

# ── Save all results ─────────────────────────────────────────────────
with open(f'{RESULTS_DIR}/eomt_all_results.json', 'w') as fp:
    json.dump(all_results, fp, indent=2, default=str)
print(f"\n✓ All results saved to {RESULTS_DIR}/eomt_all_results.json")

## 8 ‐ Print Results Tables

Load results from the JSON file and print them as formatted tables, grouped by checkpoint and temperature scaling variant.

In [ ]:
import json
import pandas as pd

# Load results
results_path = f'{RESULTS_DIR}/eomt_all_results.json'
with open(results_path) as f:
    results = json.load(f)

DS_ORDER = ['SMIYC_RA21', 'SMIYC_RO21', 'FS_LF', 'FS_Static', 'RoadAnomaly']
METHOD_ORDER = ['MSP', 'MaxLogit', 'MaxEntropy', 'RbA']

# ── Table 1: Main results (per checkpoint) ───────────────────────────
for ckpt_name, ckpt_data in results.items():
    print(f"\n{'='*80}")
    print(f"  EoMT-{ckpt_name}")
    print(f"{'='*80}")

    header = f"{'Method':<12}"
    for ds in DS_ORDER:
        header += f" | {ds[:10]:^16s}"
    print(header)

    subhdr = f"{'':12}"
    for ds in DS_ORDER:
        subhdr += f" | {'AuPRC':>7} {'FPR95':>7}"
    print(subhdr)
    print('-' * len(header))

    for m in METHOD_ORDER:
        row = f"{m:<12}"
        for ds in DS_ORDER:
            if ds in ckpt_data and m in ckpt_data[ds]:
                r = ckpt_data[ds][m]
                row += f" | {r['AuPRC']:>6.1f}% {r['FPR95']:>6.1f}%"
            else:
                row += f" | {'N/A':>7} {'N/A':>7}"
        print(row)

# ── Table 2: Temperature scaling ─────────────────────────────────────
print(f"\n\n{'='*80}")
print("  TEMPERATURE SCALING (MSP)")
print(f"{'='*80}")
for ckpt_name, ckpt_data in results.items():
    if 'temperature' not in ckpt_data:
        continue
    print(f"\n--- EoMT-{ckpt_name} ---")
    temp_data = ckpt_data['temperature']

    header = f"{'Method':<15}"
    for ds in DS_ORDER:
        header += f" | {ds[:10]:^16s}"
    print(header)

    for T_label in ['T=0.5', 'T=0.75', 'T=1.1', 'best_T']:
        row = f"{'MSP('+T_label+')':<15}" if T_label != 'best_T' else f"{'MSP(best T)':<15}"
        for ds in DS_ORDER:
            if ds in temp_data and T_label in temp_data[ds]:
                r = temp_data[ds][T_label]
                if T_label == 'best_T':
                    row += f" | {r['AuPRC']:>6.1f}% T={r['T']}"
                else:
                    row += f" | {r['AuPRC']:>6.1f}% {r['FPR95']:>6.1f}%"
            else:
                row += f" | {'N/A':>7} {'N/A':>7}"
        print(row)

## 9 ‐ Save Results

Verify that the JSON result files were saved correctly

In [ ]:
# check if results files are present
print(f"Results directory: {RESULTS_DIR}")
!ls -la "{RESULTS_DIR}/"

# Save a readable summary for easier analysis
summary_path = f'{RESULTS_DIR}/eomt_results_summary.txt'
with open(summary_path, 'w') as f:
    for ckpt_name, ckpt_data in all_results.items():
        f.write(f"\n{'='*60}\n")
        f.write(f"EoMT-{ckpt_name}\n")
        f.write(f"{'='*60}\n")
        for ds_name in DS_ORDER:
            if ds_name in ckpt_data and isinstance(ckpt_data[ds_name], dict):
                f.write(f"\n  {ds_name}:\n")
                for m, vals in ckpt_data[ds_name].items():
                    if isinstance(vals, dict) and 'AuPRC' in vals:
                        f.write(f"    {m:<15} AuPRC={vals['AuPRC']:.2f}%  FPR95={vals['FPR95']:.2f}%\n")
print(f"\n✓ Summary saved to {summary_path}")